# Statistical Outlier Detection

This notebook demonstrates statistical outlier detection algorithms using **Z-score** and **Interquartile Range (IQR)** methods, along with handling strategies (Capping, Flagging) and generating audit cleaning logs.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# Load raw dataset
df = pd.read_csv('../data/raw/customer_revenue.csv')
df.head()

## Task 1: Z-Score Outlier Detection

Detect outliers as values beyond $\pm 3$ standard deviations from the mean.

In [ ]:
df['revenue_zscore'] = np.abs(stats.zscore(df['revenue']))
z_outliers = df[df['revenue_zscore'] > 3]

print(f"Z-score outliers: {len(z_outliers)}")
z_outliers

## Task 2: IQR Outlier Detection

Detect outliers beyond $1.5 \times IQR$ from quartiles.

In [ ]:
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['is_outlier_iqr'] = (df['revenue'] < lower) | (df['revenue'] > upper)
print(f"Lower bound: {lower}, Upper bound: {upper}")
print(f"IQR Outliers: {df['is_outlier_iqr'].sum()}")

## Task 3: Cap Outliers at Boundaries

Apply capping strategy: replace extreme values with upper and lower boundary values.

In [ ]:
df['revenue_capped'] = df['revenue'].clip(lower=lower, upper=upper)

print(f"Before: min={df['revenue'].min()}, max={df['revenue'].max()}")
print(f"After: min={df['revenue_capped'].min()}, max={df['revenue_capped'].max()}")

## Task 4: Flag Outliers with Binary Column

Mark anomalies using a combined binary column without dropping rows.

In [ ]:
# Combine both Z-score and IQR flags
df['is_outlier'] = (df['is_outlier_iqr']) | (df['revenue_zscore'] > 3)

normal = df[~df['is_outlier']]
anomalies = df[df['is_outlier']]

print(f"Normal records: {len(normal)}")
print(f"Anomalies: {len(anomalies)}")

## Task 5: Create Cleaning Log

Document all outlier handling decisions in a cleaning log saved to `output/cleaning_log.csv`.

In [ ]:
cleaning_log = [{
    'column': 'revenue',
    'method': 'IQR',
    'action': 'cap',
    'threshold_lower': lower,
    'threshold_upper': upper,
    'affected_rows': int(df['is_outlier_iqr'].sum()),
    'date': pd.Timestamp.now()
}]

log_df = pd.DataFrame(cleaning_log)
os.makedirs('../output', exist_ok=True)
log_df.to_csv('../output/cleaning_log.csv', index=False)
log_df